# Evaluación MLP y deriva de covariables

Usá esta notebook para comparar bundles sobre un holdout humano exacto o revisar deriva entre cohortes compatibles. Siempre es read-only: no entrena, persiste ni promociona modelos.

| Preset | Entradas requeridas | Resultado |
|---|---|---|
| `EvaluationPreset.CHECK_ONLY` | Ninguna | Verificación del entorno |
| `EvaluationPreset.COMPARE_MODELS` | Champion, Challenger y holdout ZIP | Comparación emparejada |
| `EvaluationPreset.FILE_DRIFT` | Dos cohortes de features | Informe de deriva |
| `EvaluationPreset.POSTGRES_DRIFT` | Referencia + ventana UTC | Deriva contra telemetría read-only |
| `EvaluationPreset.CUSTOM` | Configuración completa | Comparación y deriva combinables |

**Inicio rápido recomendado:** conservá `EvaluationPreset.CHECK_ONLY`; después elegí el análisis y completá sólo su grupo `EvaluationPresetInputs`.

<details>
<summary><strong>Alcance y personalización</strong></summary>

Los tres estados aprendidos son `Normal`, `Reduced` y `Congested`; `Accident` continúa siendo humano. Los intervalos y PSI son evidencia descriptiva, no promoción automática. Consultá la [guía central de presets](../../../docs/operations/notebook-configuration.md).

</details>


In [ ]:
# Preparación del entorno: ejecutá esta celda una vez por runtime.
import importlib.util
import os
import runpy
import subprocess
from pathlib import Path

IN_COLAB = importlib.util.find_spec("google.colab") is not None
REPO_URL = "https://github.com/zgfnicolas/vaaet.git"
WORKSPACE_DIR = Path("/content/vaaet")
if IN_COLAB:
    if (WORKSPACE_DIR / ".git").is_dir():
        subprocess.check_call(["git", "-C", str(WORKSPACE_DIR), "pull", "--ff-only"])
    else:
        subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(WORKSPACE_DIR)])
else:
    candidates = [Path.cwd(), *Path.cwd().parents]
    WORKSPACE_DIR = next(
        (
            path
            for path in candidates
            if (path / "vaaet-core/pyproject.toml").is_file()
            and (path / "vaaet-persistence/pyproject.toml").is_file()
            and (path / "vaaet-ml/pyproject.toml").is_file()
        ),
        None,
    )
    if WORKSPACE_DIR is None:
        raise RuntimeError("No se encontró el workspace VAAET con core y ML.")
CORE_ROOT = WORKSPACE_DIR / "vaaet-core"
PERSISTENCE_ROOT = WORKSPACE_DIR / "vaaet-persistence"
ML_ROOT = WORKSPACE_DIR / "vaaet-ml"
REPO_ROOT = ML_ROOT
os.chdir(ML_ROOT)
BOOTSTRAP = runpy.run_path(str(ML_ROOT / "scripts" / "notebook_bootstrap.py"))
RUNTIME = BOOTSTRAP["bootstrap_notebook"](
    workspace_root=WORKSPACE_DIR,
    core_root=CORE_ROOT,
    persistence_root=PERSISTENCE_ROOT,
    ml_root=ML_ROOT,
    core_extras=('inference',),
    ml_extras=('visualization', 'database'),
    in_colab=IN_COLAB,
    framework='tensorflow',
    require_gpu=False,
)
VAAET_PACKAGE_FILE = RUNTIME.package_file
VAAET_ML_PACKAGE_FILE = RUNTIME.ml_package_file
GIT_COMMIT = RUNTIME.git_commit


In [ ]:
# Configuración del workflow: editá únicamente esta celda.
from dataclasses import replace

from vaaet_ml.workflow_presets import (
    EvaluationPreset,
    EvaluationPresetInputs,
    evaluation_preset_config,
    render_workflow_summary,
    resolve_evaluation_config,
)

SELECTED_PRESET = EvaluationPreset.CHECK_ONLY
PRESET_INPUTS = EvaluationPresetInputs()
CUSTOM_CONFIG = None
WORKFLOW_CONFIG = resolve_evaluation_config(
    SELECTED_PRESET,
    preset_inputs=PRESET_INPUTS,
    custom_config=CUSTOM_CONFIG,
)

print(render_workflow_summary(SELECTED_PRESET, WORKFLOW_CONFIG))


In [ ]:
# Imports del workflow: no edites esta celda.
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psycopg2
import sqlalchemy
import tensorflow as tf

from vaaet_persistence import load_telemetry_window
from vaaet_ml.data.database import DatabaseProfile, get_optional_database_settings
from vaaet_ml.evaluation.champion_challenger import evaluate_champion_challenger, load_evaluation_bundle, plot_champion_challenger_confusion
from vaaet_ml.evaluation.drift import build_feature_cohort, build_feature_cohort_from_raw_telemetry, compare_feature_cohorts, plot_feature_drift
from vaaet_ml.training.holdout import FileSystemHoldoutStore



## 1. Champion vs. Challenger

La notebook valida ambos manifiestos antes de cargar Keras o joblib. Si los fingerprints de holdout no coinciden, la comparación se detiene: benchmarks distintos no son A/B comparables. `direct_*` mide la salida inmediata del MLP y `final_*`, el estado posterior a calibración e histéresis. Los intervalos remuestrean clips completos, no minutos aislados.


In [ ]:
comparison = None
holdout_snapshot = None
if WORKFLOW_CONFIG.run_model_evaluation:
    holdout_path = Path(WORKFLOW_CONFIG.holdout_snapshot_path).expanduser().resolve()
    if not holdout_path.is_file():
        raise FileNotFoundError(f"No existe el holdout configurado: {holdout_path.name}")
    holdout_snapshot = FileSystemHoldoutStore(holdout_path.parent).load_snapshot(holdout_path)
    champion_bundle = load_evaluation_bundle(WORKFLOW_CONFIG.champion_bundle_dir, name="Champion")
    challenger_bundle = load_evaluation_bundle(WORKFLOW_CONFIG.challenger_bundle_dir, name="Challenger")
    comparison = evaluate_champion_challenger(
        champion_bundle, challenger_bundle, holdout_snapshot, bootstrap_samples=WORKFLOW_CONFIG.bootstrap_samples
    )
    print(f"✅ Holdout exacto: {holdout_snapshot.descriptor['fingerprint']}")
    for bundle in (champion_bundle, challenger_bundle):
        blockers = bundle.manifest["data_provenance"].get("promotion_blockers", [])
        print(
            f"{bundle.name} | revisión={bundle.manifest['model_revision']} "
            f"| blockers declarados: {blockers or 'ninguno'}"
        )
    print(comparison.summary.to_string(index=False))
    print("\nIntervalos bootstrap emparejados (95%):")
    print(comparison.bootstrap_intervals.to_string(index=False))
else:
    print("ℹ️ Comparación MLP desactivada; no se cargaron bundles ni holdouts.")


In [ ]:
if comparison is not None and holdout_snapshot is not None:
    plot_champion_challenger_confusion(comparison, holdout_snapshot.test["traffic_state"].to_numpy())
    print("\nSoporte directo del Champion:")
    print(comparison.champion.direct_support_table.to_string(index=False))
    print("\nSoporte e intervalos finales del Champion:")
    print(comparison.champion.support_table.to_string(index=False))
    print("\nSoporte directo del Challenger:")
    print(comparison.challenger.direct_support_table.to_string(index=False))
    print("\nSoporte e intervalos finales del Challenger:")
    print(comparison.challenger.support_table.to_string(index=False))


## 2. Deriva de covariables

Pregunta: ¿la cohorte operacional presenta distribuciones o calidad de las 19 features materialmente distintas de la referencia? Los valores se describen; no se convierten en un gatillo automático de reentrenamiento.


In [ ]:
drift_report = None
if WORKFLOW_CONFIG.run_drift_analysis:
    reference_frame = pd.read_csv(WORKFLOW_CONFIG.reference_feature_cohort_path)
    reference_cohort = build_feature_cohort(reference_frame, name="Referencia")
    if WORKFLOW_CONFIG.use_postgres_operational:
        database_settings = get_optional_database_settings(DatabaseProfile.TRAINING)
        if database_settings is None:
            raise RuntimeError("PostgreSQL está habilitado, pero falta el perfil training read-only.")
        raw_operational = load_telemetry_window(
            start=pd.Timestamp(WORKFLOW_CONFIG.postgres_start_utc), end=pd.Timestamp(WORKFLOW_CONFIG.postgres_end_utc),
            pipeline_run_ids=WORKFLOW_CONFIG.postgres_pipeline_run_ids, clip_ids=WORKFLOW_CONFIG.postgres_clip_ids, settings=database_settings,
        )
        operational_cohort = build_feature_cohort_from_raw_telemetry(raw_operational, name="Operacional PostgreSQL")
    else:
        operational_frame = pd.read_csv(WORKFLOW_CONFIG.operational_feature_cohort_path)
        operational_cohort = build_feature_cohort(operational_frame, name="Operacional archivo")
    drift_report = compare_feature_cohorts(reference_cohort, operational_cohort)
    print(f"✅ Referencia: {reference_cohort.profile.records} filas / {reference_cohort.profile.clips} clips")
    print(f"✅ Operacional: {operational_cohort.profile.records} filas / {operational_cohort.profile.clips} clips")
else:
    print("ℹ️ Drift desactivado; no se leyó CSV, ZIP ni PostgreSQL.")


In [ ]:
if drift_report is not None:
    print(drift_report.summary.to_string(index=False))
    selected_features = WORKFLOW_CONFIG.drift_plot_features or None
    plot_feature_drift(drift_report, features=selected_features, max_features=6)
    print("Interpretación: PSI y cambios de cuantiles priorizan revisión humana; no modifican features, umbrales ni el MLP.")


## 3. Cierre humano

Revisá primero la compatibilidad del holdout, los blockers de los manifiestos, la incertidumbre de los deltas y la procedencia de las cohortes. Si el Challenger parece mejor, la promoción sigue siendo una decisión humana separada; esta notebook no copia `.keras`, no altera punteros y no integra la Web App.

YOLO y tracking no se evalúan aquí. Ese flujo necesita cajas, clases e identidades anotadas por humanos para medir detección y seguimiento de forma válida.


In [ ]:
print("✅ Evaluación finalizada sin persistencia, promoción ni modificaciones de datos.")
if comparison is not None:
    print("➡️ Siguiente paso humano: revisar métricas, intervalos y blockers de ambos manifiestos.")
if drift_report is not None:
    print("➡️ Siguiente paso humano: investigar las features priorizadas con su cohorte, clips e intervalo declarados.")
